# Import Variables

In [1]:
import os
exec(open("./PGS_calc_param.txt").read())

# Set Up

In [2]:
import time
import subprocess
from pathlib import Path
import pandas as pd
import gzip
from glob import glob
import psutil
import re

first_start = time.time()
start = time.time()
weights_dir = Path("/home/jupyter/workspace/workspace-bucket/calculate_pgs_bed/pgs_catalog_weights")
output_dir = Path("/home/jupyter/workspace/workspace-bucket/calculate_pgs_bed/pgs_bim_matched_weights")
weights_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
###########################################################################################
# 0. Grab BIM IDs
###########################################################################################

# Load all BIM files
bim_prefix=f"/home/jupyter/workspace/vwb-aou-datasets-controlled/{VR}/wgs/short_read/snpindel/acaf_threshold/plink_bed/"

if VR == "v8":
    bim_files = glob(f"{bim_prefix}chr*.bim")
elif VR == "v9":
    bim_files = glob(f"{bim_prefix}acaf_threshold.chr*.bim")
else:
    raise ValueError(f"Unsupported VR: {VR}")


bim_ids = set()

for path in bim_files:
    ids = pd.read_csv(
        path,
        sep="\t",
        header=None,
        usecols=[1],
        dtype={1: "string"},
    ).iloc[:, 0]
    bim_ids.update(ids.dropna())
    del ids

##### track time #####
print(f"Loaded {len(bim_ids):,} variants ({time.time()-start:.1f}s)")
######################

Loaded 131,582,861 variants (139.2s)


# Download raw PGS files from PGS catalog

In [3]:
raw_files = {}
for pgs_id in PGS_ID:
    print(pgs_id)
    url = f"https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/{pgs_id}/ScoringFiles/Harmonized/{pgs_id}_hmPOS_GRCh38.txt.gz"
    raw_file = weights_dir / f"{pgs_id}_hmPOS_GRCh38.txt.gz"
    if not raw_file.exists() or raw_file.stat().st_size == 0:
        raw_file.unlink(missing_ok=True)
        print(f"Downloading {pgs_id}...")
        max_download_attempts = 6
        for attempt in range(1, max_download_attempts + 1):
            try:
                subprocess.run(
                    ["curl", "-fL", "--connect-timeout", "60", "--max-time", "3600",
                     url, "-o", str(raw_file)],
                    check=True,
                )
                break
            except subprocess.CalledProcessError as exc:
                raw_file.unlink(missing_ok=True)
                if attempt == max_download_attempts:
                    raise RuntimeError(
                        f"Download failed for {pgs_id} after {max_download_attempts} attempts"
                    ) from exc
                retry_wait = min(5 * (2 ** (attempt - 1)), 120)
                print(
                    f"{pgs_id}: download attempt {attempt}/{max_download_attempts} failed; "
                    f"retrying in {retry_wait}s ({exc})"
                )
                time.sleep(retry_wait)
            except Exception:
                raw_file.unlink(missing_ok=True)
                raise
        if not raw_file.exists() or raw_file.stat().st_size == 0:
            raw_file.unlink(missing_ok=True)
            raise RuntimeError(f"Download produced no file for {pgs_id}")
    raw_files[pgs_id] = raw_file


PGS000662
PGS000004
PGS003446
PGS002727
PGS000024
PGS002308


# Script

## Function

In [4]:
def process_pgs(pgs_id):
    inital_start = time.time()
    start = time.time()
    
    ##################################################
    # 1. Load in weights file from PGS Catalog
    ##################################################
    print("########################## Starting", pgs_id, "#############################")
    
    raw_file = weights_dir / f"{pgs_id}_hmPOS_GRCh38.txt.gz"
    # Read into enviornment 
    score_df = pd.read_csv(raw_file, sep="\t", comment="#", compression="gzip", low_memory=False)
    raw_score_length = len(score_df)

    # Use hm_inferOtherAllele where other_allele is absent or missing.
    if "other_allele" not in score_df.columns:
        if "hm_inferOtherAllele" not in score_df.columns:
            raise KeyError(
                f"{pgs_id}: neither 'other_allele' nor 'hm_inferOtherAllele' is available"
            )
        score_df["other_allele"] = score_df["hm_inferOtherAllele"]
        print(f"{pgs_id}: Other_allele column missing, used hm_inferOtherAllele")
    else:
        missing_other = (
            score_df["other_allele"].isna()
            | score_df["other_allele"].astype("string").str.strip().eq("")
        )
    
        if missing_other.any():
            if "hm_inferOtherAllele" not in score_df.columns:
                raise KeyError(
                    f"{pgs_id}: 'other_allele' has missing values, "
                    "but 'hm_inferOtherAllele' is unavailable"
                )
    
            score_df.loc[missing_other, "other_allele"] = score_df.loc[
                missing_other, "hm_inferOtherAllele"
            ]
    
        print(
            f"{pgs_id}: filled {missing_other.sum()} missing other_allele values "
            "from hm_inferOtherAllele"
        )
    
    ##################################################
    # 2.  Prepare SNP IDs to match .bim file
    ##################################################
    start = time.time()
    print("Starting harmonization...")

    ##########################################################################################
    ### Build forward and reverse IDs from PGS catalog score file
    ##########################################################################################

    score_df["chr_name_38"] = score_df["hm_chr"]
    score_df["chr_position_38"] = pd.to_numeric(
            score_df["hm_pos"],
            errors="coerce"
        ).astype("Int64")
    
    # Build forward and reverse IDs
    print("Building SNP IDs...")
    score_df['SNP_ID'] = (
        "chr" + score_df['chr_name_38'].astype(str)
        + ":" + score_df['chr_position_38'].astype(str)
        + ":" + score_df['effect_allele']
        + ":" + score_df['other_allele']
    )

    score_df['SNP_rev'] = (
        "chr" + score_df['chr_name_38'].astype(str)
        + ":" + score_df['chr_position_38'].astype(str)
        + ":" + score_df['other_allele']
        + ":" + score_df['effect_allele']
    )

    ##### track time #####
    print(f"{pgs_id}: Done building SNP IDs in ({time.time()-start:.1f}s)")
    ######################
    #########################################################################################
    ### Find allele matches from PGS score file in BIM file
    ##########################################################################################

    start = time.time()
    print(pgs_id, ": Determining matches...")

    # Check both allele orientations in one membership pass, then split the
    # Boolean results back into forward and reverse matches.
    n_variants = len(score_df)
    candidate_ids = pd.concat(
        [score_df["SNP_ID"], score_df["SNP_rev"]],
        ignore_index=True,
    )
    candidate_matches = candidate_ids.isin(bim_ids).to_numpy()
    forward = pd.Series(candidate_matches[:n_variants], index=score_df.index)
    reverse = pd.Series(candidate_matches[n_variants:], index=score_df.index)
    del candidate_ids, candidate_matches
    flip = (~forward) & reverse
    matched = forward | reverse
    final = score_df.loc[matched].copy()

    ##########################################################################################
    #### Create harmonized score file
    ##########################################################################################
    # Flip effect weights where necessary
    print(pgs_id, ": Applying allele flips...")
    
    final.loc[flip.loc[matched], "effect_weight"] *= -1
    final.loc[flip.loc[matched], "effect_allele"] = score_df.loc[flip, "other_allele"].values
    final.loc[flip.loc[matched], "other_allele"] = score_df.loc[flip, "effect_allele"].values
    final.loc[flip.loc[matched], "SNP_ID"] = score_df.loc[flip, "SNP_rev"].values

    #### Final duplicate safety drop ####
    final = final.drop_duplicates(
        subset=["SNP_ID", "effect_allele", "effect_weight"],
        keep="first",)

    #### Weird case where SNP is duplicated with two different effect weights ####
    final = (
        final.groupby(["SNP_ID", "effect_allele", "other_allele"], as_index=False)
        .agg(effect_weight=("effect_weight", "sum")))

    ##### track time #####
    print(f"{pgs_id}: Flips applied ({time.time()-start:.1f}s)")
    print(f"{pgs_id}: Total orientation matches: {int(matched.sum())}")

    #########################################################################################
    ##### Save new PRS file
    ##########################################################################################
    
    # Select columns for PLINK score file
    plink_score_df = final[['SNP_ID', 'effect_allele', 'effect_weight']]

    # Save
    plink_score_file = output_dir / f"{pgs_id}_plink_score_GRCh38_prepared.txt"
    plink_score_df.to_csv(
        plink_score_file,
        sep="\t",
        index=False,
        header=False
    )

    print(f"{pgs_id}: Saved {len(plink_score_df):,} variants to: {plink_score_file}")

    #########################################################################################
    ##### Save QC metrics file
    ##########################################################################################
    qc_file = output_dir / f"{pgs_id}_qc_metrics_GRCh38.txt"
    n_total = len(score_df)
    qc_metrics = {
        "PGS_ID": pgs_id,
        "total_pgs_variants_in_raw_score_file": raw_score_length,
        "forward_matches": int(forward.sum()), 
        "reverse_matches": int(reverse.sum()),
        "both_orientation_matches": int((forward & reverse).sum()), 
        "flipped_variants": int(flip.sum()),
        "matched_variants": int(matched.sum()), 
        "unmatched_variants": int((~matched).sum()),
        "percent_found": round(100 * matched.sum() / n_total, 2) if n_total else 0,
        "variants_saved_to_plink_score_file": len(plink_score_df),
        "plink_score_file": plink_score_file,
        "qc_file": qc_file,
        "total_runtime_minutes": round((time.time() - inital_start) / 60, 2),
    }

    pd.DataFrame([qc_metrics]).to_csv(
        qc_file,
        sep="\t",
        index=False
    )

    print(pgs_id, ": Saved QC metrics to:")
    print(qc_file)

    n_saved = len(final)

    del score_df, forward, reverse, flip, matched, plink_score_df

    ##### track time #####
    print(f"{pgs_id}: Done; Total time: {(time.time() - inital_start)/60:.1f} minutes")
    ######################

    return pgs_id, n_saved, str(plink_score_file), str(qc_file)


# Run

In [5]:
##########################################################################################
    #### Run the code using a multiprocesser 
##########################################################################################
    
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

pgs_ids = list(dict.fromkeys(PGS_ID))
results = []
errors = []

with ProcessPoolExecutor(
    max_workers=min(len(pgs_ids), 5),
    mp_context=mp.get_context("fork"),
) as executor:
    futures = {executor.submit(process_pgs, pgs_id): pgs_id for pgs_id in pgs_ids}

    for future in as_completed(futures):
        pgs_id = futures[future]
        try:
            result = future.result()
            results.append(result)
            print(f"{pgs_id}: saved {result[1]:,} matched variants to {result[2]}")
        except Exception as exc:
            errors.append((pgs_id, exc))
            print(f"{pgs_id}: FAILED: {exc}")

del bim_ids

##########################################################################################
    #### Update the param file to reflect failed jobs
##########################################################################################

# Preserve successful IDs for downstream notebook steps and keep failures separate.
PGS_failed = [pgs_id for pgs_id, _ in errors]
PGS_IDS = [pgs_id for pgs_id in PGS_ID if pgs_id not in PGS_failed]

# Update param file
param_file = Path("./PGS_calc_param.txt")

param_text = param_file.read_text()
outcome_assignments = {
    "PGS_ID_updated": PGS_IDS,
    "PGS_failed": PGS_failed,
}
for name, value in outcome_assignments.items():
    assignment = f"{name} = {value!r}"
    pattern = rf"(?m)^{re.escape(name)}\s*=.*$"
    if re.search(pattern, param_text):
        param_text = re.sub(pattern, lambda _: assignment, param_text)
    else:
        param_text = param_text.rstrip() + "\n" + assignment + "\n"
param_file.write_text(param_text)
print(f"Updated parameter file: {param_file.resolve()}")

##########################################################################################
    #### Combine the individual per-PGS QC reports and display the summary
##########################################################################################

pgs_ids_tag = "_".join(PGS_IDS)
combined_qc_file = output_dir / f"combined_qc_metrics_{pgs_ids_tag}.txt"
qc_files = [output_dir / f"{pgs_id}_qc_metrics_GRCh38.txt" for pgs_id in PGS_IDS]
combined_qc = pd.concat(
    [pd.read_csv(qc_file, sep="\t") for qc_file in qc_files],
    ignore_index=True,
)
combined_qc.to_csv(combined_qc_file, sep="\t", index=False)
print(f"Finished {len(results)} PGS jobs in  {(time.time() - first_start)/60:.1f} minutes")
print(f"Combined QC metrics saved to: {combined_qc_file}")

##########################################################################################
    #### Combine prepared PLINK score rows
##########################################################################################

combined_weights_start = time.time()

combined_weights_file = output_dir / (f"combined_plink_score_{pgs_ids_tag}_GRCh38_wide.txt.gz")

weight_frames = []
for pgs_id in PGS_IDS:
    weights_file = output_dir / f"{pgs_id}_plink_score_GRCh38_prepared.txt"
    weights = pd.read_csv(
        weights_file,
        sep="\t",
        header=None,
        names=["SNP_ID", "effect_allele", "effect_weight"],
        dtype={"SNP_ID": "string", "effect_allele": "string"},
    )
    weights["effect_allele"] = weights["effect_allele"].str.upper()
    weights["effect_weight"] = pd.to_numeric(weights["effect_weight"], errors="raise")
    weights["PGS_ID"] = pgs_id
    weight_frames.append(weights)

long_weights = pd.concat(weight_frames, ignore_index=True)
del weight_frames

# A wide PLINK score file has one allele column, so each SNP must use
# the same effect allele across all PGS files.
allele_counts = long_weights.groupby("SNP_ID")["effect_allele"].nunique()
conflicts = allele_counts[allele_counts > 1]
if not conflicts.empty:
    examples = conflicts.index[:10].tolist()
    raise ValueError(
        "Effect-allele conflicts remain across prepared PGS files. "
        f"Example SNP_IDs: {examples}"
    )

duplicates = long_weights.duplicated(["SNP_ID", "PGS_ID"], keep=False)
if duplicates.any():
    raise ValueError("Duplicate SNP_ID/PGS_ID rows found in prepared score files.")

wide_weights = long_weights.pivot(
    index=["SNP_ID", "effect_allele"],
    columns="PGS_ID",
    values="effect_weight",
)

# Keep the requested PGS column order; zero means that PGS has no weight for that SNP.
wide_weights = wide_weights.reindex(columns=PGS_IDS).fillna(0.0)
wide_weights.columns.name = None
wide_weights = wide_weights.reset_index()

wide_weights.to_csv(combined_weights_file, sep="\t", index=False, compression="gzip")

# Save by chromosome
chrom = wide_weights["SNP_ID"].str.split(":", n=1).str[0].str.removeprefix("chr")
wide_weights = wide_weights.assign(_CHROM=chrom)

chrom_weights_files = {}

for chrom, group in wide_weights.groupby("_CHROM", sort=False):
    out_file = output_dir / (
        f"combined_plink_score_{pgs_ids_tag}_chr{chrom}_GRCh38_prepared.txt.gz"
    )
    group.drop(columns="_CHROM").to_csv(out_file, sep="\t", index=False, compression="gzip")
    chrom_weights_files[chrom] = out_file
    print(f"Saved chr{chrom} wide weights: {out_file}")

print(f"Wide score file saved to: {combined_weights_file} after {(time.time() - combined_weights_start)/60:.1f} minutes")

# Add combined weights file to param file
param_file = Path("./PGS_calc_param.txt")
param_text = param_file.read_text()

name = "COMBINED_WEIGHTS_FILE"
assignment = f"{name} = {str(combined_weights_file)!r}"
pattern = rf"(?m)^{name}\s*=.*$"

if re.search(pattern, param_text):
    param_text = re.sub(pattern, lambda _: assignment, param_text)
else:
    param_text = param_text.rstrip() + "\n" + assignment + "\n"

param_file.write_text(param_text)
print(f"Updated parameter file: {param_file.resolve()}")

combined_qc.head(20)

########################## Starting########################## Starting########################## Starting########################## Starting########################## Starting     PGS000662PGS003446PGS000004PGS002727PGS000024     #################################################################################################################################################




PGS000024: Other_allele column missing, used hm_inferOtherAllele
Starting harmonization...
Building SNP IDs...
PGS000024: Done building SNP IDs in (0.0s)
PGS000024 : Determining matches...
PGS000004: filled 0 missing other_allele values from hm_inferOtherAllelePGS000662: filled 0 missing other_allele values from hm_inferOtherAllele

Starting harmonization...Starting harmonization...

Building SNP IDs...Building SNP IDs...

PGS000662: Done building SNP IDs in (0.0s)
PGS000662PGS000004: Done building SNP IDs in (0.0s) 
: Determining matches...PGS000004
 : Determining matches...
PGS003446: filled 0 missing other_all

,PGS_ID,total_pgs_variants_in_raw_score_file,forward_matches,reverse_matches,both_orientation_matches,flipped_variants,matched_variants,unmatched_variants,percent_found,variants_saved_to_plink_score_file,plink_score_file,qc_file,total_runtime_minutes
0,PGS000662,269,126,116,0,116,242,27,89.96,242,/home/jupyter/workspace/workspace-bucket/calcu...,/home/jupyter/workspace/workspace-bucket/calcu...,3.00
1,PGS000004,313,25,304,24,280,305,8,97.44,305,/home/jupyter/workspace/workspace-bucket/calcu...,/home/jupyter/workspace/workspace-bucket/calcu...,2.94
2,PGS003446,538084,134843,386471,0,386471,521314,16770,96.88,521314,/home/jupyter/workspace/workspace-bucket/calcu...,/home/jupyter/workspace/workspace-bucket/calcu...,3.65
3,PGS002727,985837,291947,693067,0,693067,985014,823,99.92,985014,/home/jupyter/workspace/workspace-bucket/calcu...,/home/jupyter/workspace/workspace-bucket/calcu...,3.72
4,PGS000024,85,0,0,0,0,0,85,0.00,0,/home/jupyter/workspace/workspace-bucket/calcu...,/home/jupyter/workspace/workspace-bucket/calcu...,2.92
5,PGS002308,1259754,346752,911603,0,911603,1258355,1399,99.89,1258355,/home/jupyter/workspace/workspace-bucket/calcu...,/home/jupyter/workspace/workspace-bucket/calcu...,3.60
